# Orignal

left facies vs right porosity

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_porosity_augmented.py
import os
import re
from typing import Dict, Tuple, List
from PIL import Image

# ---- Paths (original, non-augmented) ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_facies'
porosity_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Porosity'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_fc vs por'
os.makedirs(output_dir, exist_ok=True)

# Accept ANY prefix and OPTIONAL _augNN (so org_* files match as aug=0)
PATTERN = re.compile(r'^[A-Za-z]+_layer_(\d+).*?(?:_aug(\d+))?\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    """Return {(layer, aug): path}. If no _augNN in name -> aug=0."""
    out: Dict[Tuple[int, int], str] = {}
    skipped: List[str] = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            skipped.append(fname)
            continue
        layer = int(m.group(1))
        aug = int(m.group(2)) if m.group(2) is not None else 0
        out[(layer, aug)] = os.path.join(folder, fname)
    if skipped:
        print(f"[{os.path.basename(folder)}] skipped (pattern mismatch): {skipped[:8]}{' …' if len(skipped)>8 else ''}")
    return out

def resample_for(img: Image.Image) -> int:
    # Palette images should use NEAREST to avoid color bleed
    return Image.NEAREST if img.mode == 'P' else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    poro   = index_images(porosity_dir)

    common_keys = sorted(set(facies.keys()) & set(poro.keys()))
    missing_f   = sorted(set(poro.keys()) - set(facies.keys()))
    missing_p   = sorted(set(facies.keys()) - set(poro.keys()))

    print(f"Facies files indexed:   {len(facies)}")
    print(f"Porosity files indexed: {len(poro)}")
    print(f"Pairs to combine:       {len(common_keys)}")
    if missing_f:
        sample = [f"layer={k[0]}, aug={k[1]:02d}" for k in missing_f[:5]]
        print(f"WARNING: {len(missing_f)} porosity images lack facies pairs (e.g., {sample})")
    if missing_p:
        sample = [f"layer={k[0]}, aug={k[1]:02d}" for k in missing_p[:5]]
        print(f"WARNING: {len(missing_p)} facies images lack porosity pairs (e.g., {sample})")

    for (layer, aug) in common_keys:
        f_path = facies[(layer, aug)]
        p_path = poro[(layer, aug)]

        f_img = Image.open(f_path)
        p_img = Image.open(p_path)

        # Match heights (palette-safe for 'P' images)
        target_h = min(f_img.height, p_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=resample_for(f_img))
        if p_img.height != target_h:
            p_w = int(round(p_img.width * (target_h / p_img.height)))
            p_img = p_img.resize((p_w, target_h), resample=resample_for(p_img))

        # Side-by-side: Facies left, Porosity right
        combined_w = f_img.width + p_img.width
        canvas = Image.new('RGB', (combined_w, target_h))
        canvas.paste(f_img.convert('RGB'), (0, 0))
        canvas.paste(p_img.convert('RGB'), (f_img.width, 0))

        out_name = f"combined_layer_{layer}" + (f"_aug{aug:02d}.png" if aug > 0 else ".png")
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, format='PNG', optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Left pors vs facs

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_porosity_augmented.py
import os
import re
from typing import Dict, Tuple, List
from PIL import Image

# ---- Paths (original, non-augmented) ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_facies'
porosity_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Porosity'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_right_fc vs por'
os.makedirs(output_dir, exist_ok=True)

# Accept ANY prefix and OPTIONAL _augNN (so org_* files match as aug=0)
PATTERN = re.compile(r'^[A-Za-z]+_layer_(\d+).*?(?:_aug(\d+))?\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    """Return {(layer, aug): path}. If no _augNN in name -> aug=0."""
    out: Dict[Tuple[int, int], str] = {}
    skipped: List[str] = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            skipped.append(fname)
            continue
        layer = int(m.group(1))
        aug = int(m.group(2)) if m.group(2) is not None else 0
        out[(layer, aug)] = os.path.join(folder, fname)
    if skipped:
        print(f"[{os.path.basename(folder)}] skipped (pattern mismatch): {skipped[:8]}{' …' if len(skipped)>8 else ''}")
    return out

def resample_for(img: Image.Image) -> int:
    # Why: palette images must use NEAREST to preserve class colors
    return Image.NEAREST if img.mode == 'P' else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    poro   = index_images(porosity_dir)

    common_keys = sorted(set(facies.keys()) & set(poro.keys()))
    missing_f   = sorted(set(poro.keys()) - set(facies.keys()))
    missing_p   = sorted(set(facies.keys()) - set(poro.keys()))

    print(f"Facies files indexed:   {len(facies)}")
    print(f"Porosity files indexed: {len(poro)}")
    print(f"Pairs to combine:       {len(common_keys)}")
    if missing_f:
        sample = [f"layer={k[0]}, aug={k[1]:02d}" for k in missing_f[:5]]
        print(f"WARNING: {len(missing_f)} porosity images lack facies pairs (e.g., {sample})")
    if missing_p:
        sample = [f"layer={k[0]}, aug={k[1]:02d}" for k in missing_p[:5]]
        print(f"WARNING: {len(missing_p)} facies images lack porosity pairs (e.g., {sample})")

    for (layer, aug) in common_keys:
        f_path = facies[(layer, aug)]
        p_path = poro[(layer, aug)]

        facies_img   = Image.open(f_path)
        porosity_img = Image.open(p_path)

        # Match heights (palette-safe)
        target_h = min(facies_img.height, porosity_img.height)
        if porosity_img.height != target_h:
            p_w = int(round(porosity_img.width * (target_h / porosity_img.height)))
            porosity_img = porosity_img.resize((p_w, target_h), resample=resample_for(porosity_img))
        if facies_img.height != target_h:
            f_w = int(round(facies_img.width * (target_h / facies_img.height)))
            facies_img = facies_img.resize((f_w, target_h), resample=resample_for(facies_img))

        # ---- Paste order swapped: Porosity LEFT, Facies RIGHT ----
        combined_w = porosity_img.width + facies_img.width
        canvas = Image.new('RGB', (combined_w, target_h))
        canvas.paste(porosity_img.convert('RGB'), (0, 0))                  # left
        canvas.paste(facies_img.convert('RGB'), (porosity_img.width, 0))   # right

        out_name = f"combined_layer_{layer}" + (f"_aug{aug:02d}.png" if aug > 0 else ".png")
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, format='PNG', optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Facies Vs Vshale

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_vshale_augmented.py
import os
import re
from typing import Dict, Tuple, List
from PIL import Image

# ---- Paths ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_facies'
vshale_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_VShale'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_fc_vs_Vshale'
os.makedirs(output_dir, exist_ok=True)

# ---- Accept ANY prefix, optional _augNN ----
PATTERN = re.compile(r'^[A-Za-z]+_layer_(\d+).*?(?:_aug(\d+))?\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    """Return {(layer, aug): path}. If no _augNN -> aug=0."""
    out: Dict[Tuple[int, int], str] = {}
    skipped: List[str] = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            skipped.append(fname)
            continue
        layer = int(m.group(1))
        aug = int(m.group(2)) if m.group(2) else 0
        out[(layer, aug)] = os.path.join(folder, fname)
    if skipped:
        print(f"[{os.path.basename(folder)}] skipped (no match): {skipped[:5]}{' …' if len(skipped)>5 else ''}")
    return out

def resample_for(img: Image.Image) -> int:
    """Use NEAREST for palette PNGs to keep colors exact."""
    return Image.NEAREST if img.mode == "P" else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    vsh    = index_images(vshale_dir)

    common_keys = sorted(set(facies) & set(vsh))
    missing_f   = sorted(set(vsh) - set(facies))
    missing_v   = sorted(set(facies) - set(vsh))

    print(f"Facies indexed: {len(facies)}")
    print(f"VShale indexed: {len(vsh)}")
    print(f"Pairs to combine: {len(common_keys)}")
    if missing_f:
        print(f"WARNING: {len(missing_f)} VShale images without Facies (e.g., {missing_f[:5]})")
    if missing_v:
        print(f"WARNING: {len(missing_v)} Facies images without VShale (e.g., {missing_v[:5]})")

    for (layer, aug) in common_keys:
        f_img = Image.open(facies[(layer, aug)])
        v_img = Image.open(vsh[(layer, aug)])

        # Match heights
        target_h = min(f_img.height, v_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=resample_for(f_img))
        if v_img.height != target_h:
            v_w = int(round(v_img.width * (target_h / v_img.height)))
            v_img = v_img.resize((v_w, target_h), resample=resample_for(v_img))

        # Combine side-by-side
        combined_w = f_img.width + v_img.width
        canvas = Image.new("RGB", (combined_w, target_h))
        canvas.paste(f_img.convert("RGB"), (0, 0))
        canvas.paste(v_img.convert("RGB"), (f_img.width, 0))

        out_name = f"combined_layer_{layer}" + (f"_aug{aug:02d}.png" if aug else ".png")
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, "PNG", optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Permeability on the left and facies on the right:

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_perm_augmented.py
import os
import re
from typing import Dict, Tuple, List
from PIL import Image

# ---- Paths ----
facies_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_facies'
perm_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Perm'
output_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_left_fc_vs_Perm'
os.makedirs(output_dir, exist_ok=True)

# ---- Accept ANY prefix, optional _augNN ----
PATTERN = re.compile(r'^[A-Za-z]+_layer_(\d+).*?(?:_aug(\d+))?\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    """Return {(layer, aug): path}. If no _augNN -> aug=0."""
    out: Dict[Tuple[int, int], str] = {}
    skipped: List[str] = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            skipped.append(fname)
            continue
        layer = int(m.group(1))
        aug = int(m.group(2)) if m.group(2) else 0
        out[(layer, aug)] = os.path.join(folder, fname)
    if skipped:
        print(f"[{os.path.basename(folder)}] skipped (no match): {skipped[:5]}{' …' if len(skipped)>5 else ''}")
    return out

def resample_for(img: Image.Image) -> int:
    """Use NEAREST for palette PNGs to keep colors exact."""
    return Image.NEAREST if img.mode == "P" else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    perm   = index_images(perm_dir)

    common_keys = sorted(set(facies) & set(perm))
    missing_f   = sorted(set(perm) - set(facies))
    missing_p   = sorted(set(facies) - set(perm))

    print(f"Facies indexed: {len(facies)}")
    print(f"Perm   indexed: {len(perm)}")
    print(f"Pairs to combine: {len(common_keys)}")
    if missing_f:
        print(f"WARNING: {len(missing_f)} Perm images without Facies (e.g., {missing_f[:5]})")
    if missing_p:
        print(f"WARNING: {len(missing_p)} Facies images without Perm (e.g., {missing_p[:5]})")

    for (layer, aug) in common_keys:
        f_img = Image.open(facies[(layer, aug)])
        p_img = Image.open(perm[(layer, aug)])

        # Match heights
        target_h = min(f_img.height, p_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=resample_for(f_img))
        if p_img.height != target_h:
            p_w = int(round(p_img.width * (target_h / p_img.height)))
            p_img = p_img.resize((p_w, target_h), resample=resample_for(p_img))

        # ---- Facies LEFT, Perm RIGHT ----
        combined_w = f_img.width + p_img.width
        canvas = Image.new("RGB", (combined_w, target_h))
        canvas.paste(f_img.convert("RGB"), (0, 0))            # left: Facies
        canvas.paste(p_img.convert("RGB"), (f_img.width, 0))  # right: Perm

        out_name = f"combined_layer_{layer}" + (f"_aug{aug:02d}.png" if aug else ".png")
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, "PNG", optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Permeability on the right and facies on the left:

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_perm_augmented.py
import os
import re
from typing import Dict, Tuple, List
from PIL import Image

# ---- Paths ----
facies_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_facies'
perm_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Perm'
output_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_left_Perm_vs_fc'
os.makedirs(output_dir, exist_ok=True)

# ---- Accept ANY prefix, optional _augNN ----
PATTERN = re.compile(r'^[A-Za-z]+_layer_(\d+).*?(?:_aug(\d+))?\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    """Return {(layer, aug): path}. If no _augNN -> aug=0."""
    out: Dict[Tuple[int, int], str] = {}
    skipped: List[str] = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            skipped.append(fname)
            continue
        layer = int(m.group(1))
        aug = int(m.group(2)) if m.group(2) else 0
        out[(layer, aug)] = os.path.join(folder, fname)
    if skipped:
        print(f"[{os.path.basename(folder)}] skipped (no match): {skipped[:5]}{' …' if len(skipped)>5 else ''}")
    return out

def resample_for(img: Image.Image) -> int:
    """Use NEAREST for palette PNGs to keep colors exact."""
    return Image.NEAREST if img.mode == "P" else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    perm   = index_images(perm_dir)

    common_keys = sorted(set(facies) & set(perm))
    missing_f   = sorted(set(perm) - set(facies))
    missing_p   = sorted(set(facies) - set(perm))

    print(f"Facies indexed: {len(facies)}")
    print(f"Perm   indexed: {len(perm)}")
    print(f"Pairs to combine: {len(common_keys)}")
    if missing_f:
        print(f"WARNING: {len(missing_f)} Perm images without Facies (e.g., {missing_f[:5]})")
    if missing_p:
        print(f"WARNING: {len(missing_p)} Facies images without Perm (e.g., {missing_p[:5]})")

    for (layer, aug) in common_keys:
        f_img = Image.open(facies[(layer, aug)])
        p_img = Image.open(perm[(layer, aug)])

        # Match heights
        target_h = min(f_img.height, p_img.height)
        if p_img.height != target_h:
            p_w = int(round(p_img.width * (target_h / p_img.height)))
            p_img = p_img.resize((p_w, target_h), resample=resample_for(p_img))
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=resample_for(f_img))

        # ---- Perm LEFT, Facies RIGHT ----
        combined_w = p_img.width + f_img.width
        canvas = Image.new("RGB", (combined_w, target_h))
        canvas.paste(p_img.convert("RGB"), (0, 0))            # LEFT = Permeability
        canvas.paste(f_img.convert("RGB"), (p_img.width, 0))  # RIGHT = Facies

        out_name = f"combined_layer_{layer}" + (f"_aug{aug:02d}.png" if aug else ".png")
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, "PNG", optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Pors vs VShales

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_vshale_augmented.py
import os
import re
from typing import Dict, Tuple, List
from PIL import Image

# ---- Paths ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Porosity'
vshale_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_VShale'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_Pors_vs_Vshale'
os.makedirs(output_dir, exist_ok=True)

# ---- Accept ANY prefix, optional _augNN ----
PATTERN = re.compile(r'^[A-Za-z]+_layer_(\d+).*?(?:_aug(\d+))?\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    """Return {(layer, aug): path}. If no _augNN -> aug=0."""
    out: Dict[Tuple[int, int], str] = {}
    skipped: List[str] = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            skipped.append(fname)
            continue
        layer = int(m.group(1))
        aug = int(m.group(2)) if m.group(2) else 0
        out[(layer, aug)] = os.path.join(folder, fname)
    if skipped:
        print(f"[{os.path.basename(folder)}] skipped (no match): {skipped[:5]}{' …' if len(skipped)>5 else ''}")
    return out

def resample_for(img: Image.Image) -> int:
    """Use NEAREST for palette PNGs to keep colors exact."""
    return Image.NEAREST if img.mode == "P" else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    vsh    = index_images(vshale_dir)

    common_keys = sorted(set(facies) & set(vsh))
    missing_f   = sorted(set(vsh) - set(facies))
    missing_v   = sorted(set(facies) - set(vsh))

    print(f"Facies indexed: {len(facies)}")
    print(f"VShale indexed: {len(vsh)}")
    print(f"Pairs to combine: {len(common_keys)}")
    if missing_f:
        print(f"WARNING: {len(missing_f)} VShale images without Facies (e.g., {missing_f[:5]})")
    if missing_v:
        print(f"WARNING: {len(missing_v)} Facies images without VShale (e.g., {missing_v[:5]})")

    for (layer, aug) in common_keys:
        f_img = Image.open(facies[(layer, aug)])
        v_img = Image.open(vsh[(layer, aug)])

        # Match heights
        target_h = min(f_img.height, v_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=resample_for(f_img))
        if v_img.height != target_h:
            v_w = int(round(v_img.width * (target_h / v_img.height)))
            v_img = v_img.resize((v_w, target_h), resample=resample_for(v_img))

        # Combine side-by-side
        combined_w = f_img.width + v_img.width
        canvas = Image.new("RGB", (combined_w, target_h))
        canvas.paste(f_img.convert("RGB"), (0, 0))
        canvas.paste(v_img.convert("RGB"), (f_img.width, 0))

        out_name = f"combined_layer_{layer}" + (f"_aug{aug:02d}.png" if aug else ".png")
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, "PNG", optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Augented

left facies vs right porosity


In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_porosity_augmented.py
import os
import re
from typing import Dict, Tuple
from PIL import Image

# ---- Paths (use your folders) ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Facies_augmented'
porosity_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Porosity_augmented'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Aug_fc vs por'
os.makedirs(output_dir, exist_ok=True)

# ---- Filename parser: Facies/Porosity_layer_<LAYER>_*_aug<NN>.png ----
PATTERN = re.compile(r'^(?:Facies|Porosity|Poro)_layer_(\d+).*_aug(\d+)\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    out: Dict[Tuple[int, int], str] = {}
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            continue
        layer = int(m.group(1))
        aug   = int(m.group(2))
        out[(layer, aug)] = os.path.join(folder, fname)
    return out

def main() -> None:
    facies = index_images(facies_dir)
    poro   = index_images(porosity_dir)

    common_keys = sorted(set(facies.keys()) & set(poro.keys()))
    missing_f  = sorted(set(poro.keys()) - set(facies.keys()))
    missing_p  = sorted(set(facies.keys()) - set(poro.keys()))

    print(f"Facies files indexed:   {len(facies)}")
    print(f"Porosity files indexed: {len(poro)}")
    print(f"Pairs to combine:       {len(common_keys)}")
    if missing_f:
        print(f"WARNING: {len(missing_f)} porosity images lack facies pairs (e.g., {missing_f[:5]})")
    if missing_p:
        print(f"WARNING: {len(missing_p)} facies images lack porosity pairs (e.g., {missing_p[:5]})")

    for (layer, aug) in common_keys:
        f_path = facies[(layer, aug)]
        p_path = poro[(layer, aug)]

        f_img = Image.open(f_path)
        p_img = Image.open(p_path)

        # Resize to same height using class-safe interpolation for facies
        target_h = min(f_img.height, p_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=Image.NEAREST)  # preserve palette edges
        if p_img.height != target_h:
            p_w = int(round(p_img.width * (target_h / p_img.height)))
            p_img = p_img.resize((p_w, target_h), resample=Image.LANCZOS)

        combined_w = f_img.width + p_img.width
        canvas = Image.new('RGB', (combined_w, target_h))
        canvas.paste(f_img.convert('RGB'), (0, 0))
        canvas.paste(p_img.convert('RGB'), (f_img.width, 0))

        out_name = f"combined_layer_{layer}_aug{aug:02}.png"
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, format='PNG', optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Left pors vs facs

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_porosity_augmented.py
import os
import re
from typing import Dict, Tuple
from PIL import Image

# ---- Paths (use your folders) ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Facies_augmented'
porosity_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Porosity_augmented'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Aug_right_fc vs por'
os.makedirs(output_dir, exist_ok=True)

# ---- Filename parser: Facies/Porosity_layer_<LAYER>_*_aug<NN>.png ----
PATTERN = re.compile(r'^(?:Facies|Porosity|Poro)_layer_(\d+).*_aug(\d+)\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    out: Dict[Tuple[int, int], str] = {}
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            continue
        layer = int(m.group(1))
        aug   = int(m.group(2))
        out[(layer, aug)] = os.path.join(folder, fname)
    return out

def main() -> None:
    facies = index_images(facies_dir)
    poro   = index_images(porosity_dir)

    common_keys = sorted(set(facies.keys()) & set(poro.keys()))
    missing_f  = sorted(set(poro.keys()) - set(facies.keys()))
    missing_p  = sorted(set(facies.keys()) - set(poro.keys()))

    print(f"Facies files indexed:   {len(facies)}")
    print(f"Porosity files indexed: {len(poro)}")
    print(f"Pairs to combine:       {len(common_keys)}")
    if missing_f:
        print(f"WARNING: {len(missing_f)} porosity images lack facies pairs (e.g., {missing_f[:5]})")
    if missing_p:
        print(f"WARNING: {len(missing_p)} facies images lack porosity pairs (e.g., {missing_p[:5]})")

    for (layer, aug) in common_keys:
        f_path = facies[(layer, aug)]
        p_path = poro[(layer, aug)]

        f_img = Image.open(f_path)
        p_img = Image.open(p_path)

        # Resize to same height using class-safe interpolation for facies
        target_h = min(f_img.height, p_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=Image.NEAREST)  # preserve palette edges
        if p_img.height != target_h:
            p_w = int(round(p_img.width * (target_h / p_img.height)))
            p_img = p_img.resize((p_w, target_h), resample=Image.LANCZOS)

        # --- Swapped order: Porosity LEFT, Facies RIGHT ---
        combined_w = f_img.width + p_img.width
        canvas = Image.new('RGB', (combined_w, target_h))
        canvas.paste(p_img.convert('RGB'), (0, 0))  # Porosity first
        canvas.paste(f_img.convert('RGB'), (p_img.width, 0))  # Facies next

        out_name = f"combined_layer_{layer}_aug{aug:02}.png"
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, format='PNG', optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


Facieson the left and facies on the right:

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_porosity_augmented.py
import os
import re
from typing import Dict, Tuple, List
from PIL import Image

# ---- Paths ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Facies_augmented'
vshale_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/VShale_augmented'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Aug_fc vs Vshale'
os.makedirs(output_dir, exist_ok=True)

# Accept ANY prefix and OPTIONAL _augNN
PATTERN = re.compile(r'^[A-Za-z]+_layer_(\d+).*?(?:_aug(\d+))?\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    """Map (layer, aug) -> file; aug=0 when absent."""
    out: Dict[Tuple[int, int], str] = {}
    skipped: List[str] = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            skipped.append(fname)
            continue
        layer = int(m.group(1))
        aug   = int(m.group(2)) if m.group(2) is not None else 0
        out[(layer, aug)] = os.path.join(folder, fname)
    if skipped:
        print(f"[{os.path.basename(folder)}] skipped (pattern mismatch): {skipped[:5]}{' …' if len(skipped)>5 else ''}")
    return out

def resample_for(img: Image.Image) -> int:
    # Keep class colors if it's a palette PNG
    return Image.NEAREST if img.mode == 'P' else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    vsh    = index_images(vshale_dir)

    common_keys = sorted(set(facies.keys()) & set(vsh.keys()))
    missing_f   = sorted(set(vsh.keys()) - set(facies.keys()))
    missing_vsh = sorted(set(facies.keys()) - set(vsh.keys()))

    print(f"Facies files indexed: {len(facies)}")
    print(f"VShale files indexed: {len(vsh)}")
    print(f"Pairs to combine:     {len(common_keys)}")
    if missing_f:
        sample = [f"layer={k[0]}, aug={k[1]:02d}" for k in missing_f[:5]]
        print(f"WARNING: {len(missing_f)} VShale images lack Facies pairs (e.g., {sample})")
    if missing_vsh:
        sample = [f"layer={k[0]}, aug={k[1]:02d}" for k in missing_vsh[:5]]
        print(f"WARNING: {len(missing_vsh)} Facies images lack VShale pairs (e.g., {sample})")

    for (layer, aug) in common_keys:
        f_path = facies[(layer, aug)]
        v_path = vsh[(layer, aug)]

        f_img = Image.open(f_path)
        v_img = Image.open(v_path)

        # Resize to same height
        target_h = min(f_img.height, v_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=resample_for(f_img))
        if v_img.height != target_h:
            v_w = int(round(v_img.width * (target_h / v_img.height)))
            v_img = v_img.resize((v_w, target_h), resample=resample_for(v_img))

        # Side-by-side: Facies left, VShale right
        combined_w = f_img.width + v_img.width
        canvas = Image.new('RGB', (combined_w, target_h))
        canvas.paste(f_img.convert('RGB'), (0, 0))
        canvas.paste(v_img.convert('RGB'), (f_img.width, 0))

        # Include _augNN only when aug>0
        out_name = f"combined_layer_{layer}" + (f"_aug{aug:02d}.png" if aug > 0 else ".png")
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, format='PNG', optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Pors vs VShales

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_porosity_augmented.py
import os
import re
from typing import Dict, Tuple, List
from PIL import Image

# ---- Paths ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Porosity_augmented'
vshale_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/VShale_augmented'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Aug_Pors vs Vshale'
os.makedirs(output_dir, exist_ok=True)

# Accept ANY prefix and OPTIONAL _augNN
PATTERN = re.compile(r'^[A-Za-z]+_layer_(\d+).*?(?:_aug(\d+))?\.png$', re.IGNORECASE)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    """Map (layer, aug) -> file; aug=0 when absent."""
    out: Dict[Tuple[int, int], str] = {}
    skipped: List[str] = []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            skipped.append(fname)
            continue
        layer = int(m.group(1))
        aug   = int(m.group(2)) if m.group(2) is not None else 0
        out[(layer, aug)] = os.path.join(folder, fname)
    if skipped:
        print(f"[{os.path.basename(folder)}] skipped (pattern mismatch): {skipped[:5]}{' …' if len(skipped)>5 else ''}")
    return out

def resample_for(img: Image.Image) -> int:
    # Keep class colors if it's a palette PNG
    return Image.NEAREST if img.mode == 'P' else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    vsh    = index_images(vshale_dir)

    common_keys = sorted(set(facies.keys()) & set(vsh.keys()))
    missing_f   = sorted(set(vsh.keys()) - set(facies.keys()))
    missing_vsh = sorted(set(facies.keys()) - set(vsh.keys()))

    print(f"Facies files indexed: {len(facies)}")
    print(f"VShale files indexed: {len(vsh)}")
    print(f"Pairs to combine:     {len(common_keys)}")
    if missing_f:
        sample = [f"layer={k[0]}, aug={k[1]:02d}" for k in missing_f[:5]]
        print(f"WARNING: {len(missing_f)} VShale images lack Facies pairs (e.g., {sample})")
    if missing_vsh:
        sample = [f"layer={k[0]}, aug={k[1]:02d}" for k in missing_vsh[:5]]
        print(f"WARNING: {len(missing_vsh)} Facies images lack VShale pairs (e.g., {sample})")

    for (layer, aug) in common_keys:
        f_path = facies[(layer, aug)]
        v_path = vsh[(layer, aug)]

        f_img = Image.open(f_path)
        v_img = Image.open(v_path)

        # Resize to same height
        target_h = min(f_img.height, v_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=resample_for(f_img))
        if v_img.height != target_h:
            v_w = int(round(v_img.width * (target_h / v_img.height)))
            v_img = v_img.resize((v_w, target_h), resample=resample_for(v_img))

        # Side-by-side: Facies left, VShale right
        combined_w = f_img.width + v_img.width
        canvas = Image.new('RGB', (combined_w, target_h))
        canvas.paste(f_img.convert('RGB'), (0, 0))
        canvas.paste(v_img.convert('RGB'), (f_img.width, 0))

        # Include _augNN only when aug>0
        out_name = f"combined_layer_{layer}" + (f"_aug{aug:02d}.png" if aug > 0 else ".png")
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, format='PNG', optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Perm on the left and facies on the right:

In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_perm_aug_left.py
import os
import re
from typing import Dict, Tuple
from PIL import Image

# ---- Paths ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Facies_augmented'
perm_dir     = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Perm_augmented'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Aug_left_fc_vs_perm'
os.makedirs(output_dir, exist_ok=True)

# Allow Facies / Porosity / Poro / Permeability prefixes
PATTERN = re.compile(
    r'^(?:Facies|Porosity|Poro|Permeability)_layer_(\d+).*_aug(\d+)\.png$',
    re.IGNORECASE,
)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    out: Dict[Tuple[int, int], str] = {}
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            continue
        layer = int(m.group(1))
        aug   = int(m.group(2))
        out[(layer, aug)] = os.path.join(folder, fname)
    return out

def resample_for(img: Image.Image) -> int:
    return Image.NEAREST if img.mode == 'P' else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    perm   = index_images(perm_dir)

    common_keys = sorted(set(facies.keys()) & set(perm.keys()))
    missing_f   = sorted(set(perm.keys()) - set(facies.keys()))
    missing_p   = sorted(set(facies.keys()) - set(perm.keys()))

    print(f"Facies files indexed: {len(facies)}")
    print(f"Perm   files indexed: {len(perm)}")
    print(f"Pairs to combine:     {len(common_keys)}")
    if missing_f:
        print(f"WARNING: {len(missing_f)} Perm images lack Facies pairs (e.g., {missing_f[:5]})")
    if missing_p:
        print(f"WARNING: {len(missing_p)} Facies images lack Perm pairs (e.g., {missing_p[:5]})")

    for (layer, aug) in common_keys:
        f_img = Image.open(facies[(layer, aug)])
        p_img = Image.open(perm[(layer, aug)])

        target_h = min(f_img.height, p_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=resample_for(f_img))
        if p_img.height != target_h:
            p_w = int(round(p_img.width * (target_h / p_img.height)))
            p_img = p_img.resize((p_w, target_h), resample=resample_for(p_img))

        # Facies LEFT, Perm RIGHT
        combined_w = f_img.width + p_img.width
        canvas = Image.new('RGB', (combined_w, target_h))
        canvas.paste(f_img.convert('RGB'), (0, 0))
        canvas.paste(p_img.convert('RGB'), (f_img.width, 0))

        out_name = f"combined_layer_{layer}_aug{aug:02}.png"
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, format='PNG', optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


In [ ]:
# /home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/combine_facies_perm_aug_right.py
import os
import re
from typing import Dict, Tuple
from PIL import Image

# ---- Paths ----
facies_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Facies_augmented'
perm_dir     = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Perm_augmented'
output_dir   = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Aug_right_fc_vs_perm'
os.makedirs(output_dir, exist_ok=True)

PATTERN = re.compile(
    r'^(?:Facies|Porosity|Poro|Permeability)_layer_(\d+).*_aug(\d+)\.png$',
    re.IGNORECASE,
)

def index_images(folder: str) -> Dict[Tuple[int, int], str]:
    out: Dict[Tuple[int, int], str] = {}
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith('.png'):
            continue
        m = PATTERN.match(fname)
        if not m:
            continue
        layer = int(m.group(1))
        aug   = int(m.group(2))
        out[(layer, aug)] = os.path.join(folder, fname)
    return out

def resample_for(img: Image.Image) -> int:
    return Image.NEAREST if img.mode == 'P' else Image.LANCZOS

def main() -> None:
    facies = index_images(facies_dir)
    perm   = index_images(perm_dir)

    common_keys = sorted(set(facies.keys()) & set(perm.keys()))
    missing_f   = sorted(set(perm.keys()) - set(facies.keys()))
    missing_p   = sorted(set(facies.keys()) - set(perm.keys()))

    print(f"Facies files indexed: {len(facies)}")
    print(f"Perm   files indexed: {len(perm)}")
    print(f"Pairs to combine:     {len(common_keys)}")
    if missing_f:
        print(f"WARNING: {len(missing_f)} Perm images lack Facies pairs (e.g., {missing_f[:5]})")
    if missing_p:
        print(f"WARNING: {len(missing_p)} Facies images lack Perm pairs (e.g., {missing_p[:5]})")

    for (layer, aug) in common_keys:
        f_img = Image.open(facies[(layer, aug)])
        p_img = Image.open(perm[(layer, aug)])

        target_h = min(f_img.height, p_img.height)
        if f_img.height != target_h:
            f_w = int(round(f_img.width * (target_h / f_img.height)))
            f_img = f_img.resize((f_w, target_h), resample=resample_for(f_img))
        if p_img.height != target_h:
            p_w = int(round(p_img.width * (target_h / p_img.height)))
            p_img = p_img.resize((p_w, target_h), resample=resample_for(p_img))

        # Perm LEFT, Facies RIGHT
        combined_w = p_img.width + f_img.width
        canvas = Image.new('RGB', (combined_w, target_h))
        canvas.paste(p_img.convert('RGB'), (0, 0))
        canvas.paste(f_img.convert('RGB'), (p_img.width, 0))

        out_name = f"combined_layer_{layer}_aug{aug:02}.png"
        out_path = os.path.join(output_dir, out_name)
        canvas.save(out_path, format='PNG', optimize=True, dpi=(300, 300))
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


# Spilit code

In [ ]:
import os
import random
import shutil

# Paths
original_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/org_right_Perm_vs_fc'
augmented_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/Aug_right_fc_vs_perm'
output_dir = '/home/g202103050/Documents/multi facies 2D Model/2D Yemen Model/right_fc vs perm_dataset_split/'

# Create output directories
train_dir = os.path.join(output_dir, 'train')
val_dir = os.path.join(output_dir, 'val')
test_dir = os.path.join(output_dir, 'test')

for directory in [train_dir, val_dir, test_dir]:
    os.makedirs(directory, exist_ok=True)

# Load all image file paths
original_images = [os.path.join(original_dir, f) for f in os.listdir(original_dir) if f.endswith('.png')]
augmented_images = [os.path.join(augmented_dir, f) for f in os.listdir(augmented_dir) if f.endswith('.png')]

# Combine original and augmented images
all_images = original_images + augmented_images
random.shuffle(all_images)

# Calculate split sizes
total_images = len(all_images)
train_count = int(0.7 * total_images)
val_count = int(0.15 * total_images)
test_count = total_images - train_count - val_count  # Ensure all are used

# Split datasets
train_images = all_images[:train_count]
val_images = all_images[train_count:train_count + val_count]
test_images = all_images[train_count + val_count:]

# Function to copy and rename images
def copy_and_rename_images(image_list, target_dir):
    for i, img_path in enumerate(image_list, start=1):  # Start counting from 1
        save_path = os.path.join(target_dir, f"{i}.png")
        shutil.copy(img_path, save_path)

# Copy images to respective folders with sequential naming
copy_and_rename_images(train_images, train_dir)
copy_and_rename_images(val_images, val_dir)
copy_and_rename_images(test_images, test_dir)

# Print results
print(f"Training set: {len(train_images)} images")
print(f"Validation set: {len(val_images)} images")
print(f"Testing set: {len(test_images)} images")
